# Instructions

You will use this notebook to submit your final model for evaluation on a hidden test set.  Here are some key notes:
* ONLY edit the first cell below with the submission tag.  Do not add cells.  Do not delete the submission tag (or add it to any other cells)
* That cell should run from "start to finish".  It should include modular code (with good data science practice, using tools like `.pipe` and `.assign` and helper functions) and comments.
* You may not use any libraries beyond the ones loaded below.
* You may not filter or drop anything from the test set
* Follow the general structure: create helper functions for feature engineering, pipe them identically to your train and test set, and then train the model.
* The test set only includes homes between \\$500 (inclusive) and  \\$50,000,000 (inclusive).

Upload this file to gradescope as a Python Notebook (not a zip file; not with any ancilary files) to the "Project 3 Phase 2 Final Model Evaluation" assignment)

In [4]:
# SUBMISSION CELL
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import zipfile
import os

# Do NOT change the first three functions:
def log_transforms(df):
    """
    Input a DataFrame df with columns Sale Price and Building Square Feet
    Returns a new DataFrame that adds the log of each of these
    """
    return df.assign(**{
        "Log Sale Price": np.log(df["Sale Price"]),
        "Log Building Square Feet": np.log(df["Building Square Feet"])
    })

def add_intercept(df):
    """
    Input a DataFrame df.
    Returns a new DataFrame with an intercept column
    """
    return df.assign(intercept = 1)


def select_vars(df, varlist):
    """
    Filters down DataFrame df to just the columns in varlist,
    which will be used for prediction
    """
    return df[varlist]

# You can edit this one if you'd like to change how you filter data
def filter_training_data(df):
    """
    Filters the training data.
    Removes implausible sale prices, outlier building sizes, and
    non-arms-length transactions (Pure Market Filter == 0).
    """
    filtered_df = df[
        (df['Sale Price'] >= 500) &
        (df['Sale Price'] <= 50000000) &
        (df['Building Square Feet'] >= 200) &
        (df['Building Square Feet'] <= 15000) &
        (df['Pure Market Filter'] == 1)
    ].copy()
    return filtered_df


# Helper functions

def add_bedrm(df):
    """Adds num_bedrm column by parsing the Description text."""
    def get_bedrm(desc):
        details = desc.split("It has a total of ")[1]
        pieces = details.split(", ")
        return float(pieces[1].split()[0])
    return df.assign(num_bedrm=df["Description"].apply(get_bedrm))

def add_num_bath(df):
    """Adds num_bath column by parsing the Description text."""
    def get_bath(desc):
        details = desc.split("It has a total of ")[1]
        pieces = details.split(", ")
        return float(pieces[2].split()[1])
    return df.assign(num_bath=df["Description"].apply(get_bath))

def id_expensive(df):
    """Adds in_expensive: 1.0 if neighborhood is in top 10 by median sale price."""
    median_by_nbhd = df.groupby("Neighborhood Code")["Sale Price"].median()
    top10 = list(median_by_nbhd.nlargest(10).index)
    return df.assign(in_expensive=np.where(
        df["Neighborhood Code"].isin(top10), 1.0, 0.0
    ))

def expand_wall_material(df):
    """Creates Wall Material Cat dummies (Masonry is reference category)."""
    wall_map = {1: "Wood", 2: "Masonry", 3: "Wood&Masonry", 4: "Stucco"}
    df = df.assign(**{"Wall Material Cat": df["Wall Material"].map(wall_map)})
    for cat in ["Stucco", "Wood", "Wood&Masonry"]:
        col = f"Wall Material Cat_{cat}"
        df = df.assign(**{col: (df["Wall Material Cat"] == cat).astype(int)})
    return df

def add_log_land(df):
    """Adds Log Land Square Feet (clipping at 1 to avoid log(0))."""
    return df.assign(**{
        "Log Land Square Feet": np.log(df["Land Square Feet"].clip(lower=1))
    })

def add_log_estimates(df):
    """Adds log of the assessor's prior-year building and land estimates."""
    return df.assign(**{
        "Log Estimate (Building)": np.log(df["Estimate (Building)"].clip(lower=1)),
        "Log Estimate (Land)":     np.log(df["Estimate (Land)"].clip(lower=1))
    })

def add_central_air_ind(df):
    """Adds is_central_air: 1 if Central Air == 1 (yes), 0 otherwise."""
    return df.assign(is_central_air=(df["Central Air"] == 1).astype(int))

def add_age_features(df):
    """Adds Age squared to capture non-linear age/price relationship."""
    return df.assign(age_sq=df["Age"] ** 2)

def add_quality_features(df):
    """Adds binary quality/design indicators."""
    return df.assign(
        is_good_repair       =(df["Repair Condition"] == 1).astype(int),
        is_architect_design  =(df["Design Plan"]       == 1).astype(int),
        has_cathedral_ceiling=(df["Cathedral Ceiling"] == 1).astype(int)
    )

def add_basement_features(df):
    """Adds has_full_basement: 1 if Basement == 1 (Full)."""
    return df.assign(has_full_basement=(df["Basement"] == 1).astype(int))


def train_and_predict(train_df, test_df):
    """The full pipeline. Trains a multi-variable OLS regression on
    engineered features and returns predictions for the test set."""
    train_filtered = filter_training_data(train_df)

    def engineer(df):
        return (df
                .pipe(log_transforms)
                .pipe(add_intercept)
                .pipe(add_bedrm)
                .pipe(add_num_bath)
                .pipe(id_expensive)
                .pipe(expand_wall_material)
                .pipe(add_log_land)
                .pipe(add_log_estimates)
                .pipe(add_central_air_ind)
                .pipe(add_age_features)
                .pipe(add_quality_features)
                .pipe(add_basement_features))

    train_df = engineer(train_filtered)
    test_df  = engineer(test_df)

    varlist = [
        "intercept",
        "Log Building Square Feet",
        "Log Land Square Feet",
        "Log Estimate (Building)",
        "Log Estimate (Land)",
        "num_bedrm",
        "num_bath",
        "Age",
        "age_sq",
        "Fireplaces",
        "in_expensive",
        "Wall Material Cat_Stucco",
        "Wall Material Cat_Wood",
        "Wall Material Cat_Wood&Masonry",
        "is_central_air",
        "Garage Indicator",
        "is_good_repair",
        "is_architect_design",
        "has_cathedral_ceiling",
        "has_full_basement",
    ]

    X_train = select_vars(train_df, varlist)
    X_test  = select_vars(test_df,  varlist)
    y_train = train_df["Log Sale Price"]
    model   = sm.OLS(y_train, X_train).fit()
    return model.predict(X_test)


## Checking Your Code Works

You will have to upload this file to Gradescope to get the MSE on the hidden test set.  You will only have about 5 opportunities to submit, so you may want to make sure your pipeline actually works first.  The cell below simulates the submission process, working only off training data.  It:
- Takes the training data set you have, and splits it into a local train and test split (note that this is not the same as the test split I'll be using: you don't get to see that set!).
- It runs `train_and_predict` and shows the MSE on both your local test and train split.  
- If your above pipeline works, you should see two MSE's printed out (so you might run it before editing anything above to see that it actually works!).  These are a rough proxy for the MSE you'll get when you submit.  If your local test MSE is much bigger than your train MSE, you may well be overfitting by quite a lot!

In [5]:
# DO NOT CHANGE THIS CELL

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# Load the training data 
with zipfile.ZipFile('cook_county_data.zip') as item:
    with item.open("cook_county_train.csv") as f:
        full_train_df = pd.read_csv(f)

# Split the data into a local training and testing set
# NOTE: this is NOT the same test set that I will be using
# I will be testing on data you have never had access to
train_split, test_split = train_test_split(full_train_df, test_size=0.2, random_state=42)

# Only use the filters for eval_test_split that I used on the Gradescope test
eval_test_split = test_split[(test_split['Sale Price'] >= 500) 
                             & (test_split['Sale Price'] <= 50000000)].copy()

# This filters the train set however you want
# (based on your filter_training_data function)
eval_train_split = filter_training_data(train_split)


# Generate predictions using your the previous cell
test_predictions = train_and_predict(train_split, eval_test_split)
train_predictions = train_and_predict(train_split, eval_train_split)

# Calculate the true Log Sale Price for evaluation
y_true_test = np.log(eval_test_split['Sale Price'])
y_true_train = np.log(eval_train_split['Sale Price'])

# Calculate and print the Mean Squared Error
mse_test = mean_squared_error(y_true_test, test_predictions)
mse_train = mean_squared_error(y_true_train, train_predictions)

print("Local Training Split MSE: ",  mse_train)
print("Local Testing Split MSE: ", mse_test)

Local Training Split MSE:  0.35611373235938276
Local Testing Split MSE:  0.41236725945874797
